# Lecture: Serving Models Like a Restaurant Kitchen

Read this before opening notebook.ipynb. Each vocabulary term below gets its own section
(marked §) with a picture or worked numbers — skim the index, study the ones that feel fuzzy.

## Learning goals

- Walk a request through the gateway in order: auth → limit → queue → cache → router → backend (§4–§6).
- Explain what each layer rescues: abuse, bursts, overload, repeats, throughput, debugging (§4–§7).
- Read a per-request ledger and say which status (200/401/429) each layer produces (§8 map + trace).


## The big idea

A raw model server answers one prompt at a time and collapses under real traffic. A production **gateway** is a pipeline of cheap guards in front of it. **Auth** (who are you?) rejects strangers with 401 before any work. A **rate limit** (how many have you asked for? — lab budget 5 per key) sheds excess with 429. **Admission / queue** (is the kitchen full? — lab bound 4 concurrent) sheds overflow with 429 instead of crashing with 500. A **cache** (did we answer this exactly before?) returns repeats for free. The **router** sends each request to the right **backend** (LLM 8 ms, batch, vision 5 ms). **Batching** cooks several orders in one oven trip by sharing the fixed warm-up cost. **Metrics + traces** are the receipts: per-route counters, latencies, and per-request IDs — and the per-request **ledger** (one CSV row each) proves every claim.

Real-world analogy: a busy restaurant. The host checks reservations (auth), caps parties per night (limit), seats only what the dining room holds and turns the rest away politely (admission, never a kitchen fire = 500), serves the daily special from memory (cache), routes sushi vs grill orders to the right chef (router), bakes four pizzas at once (batch), and keeps the ticket rail plus receipts (traces + metrics) so a bad review can be investigated order by order.

This lecture simulates everything in plain Python — no servers, no ports, no network. Counts and timings below are small functions of the lab's policy numbers (1 token, budget 5, 4 slots, 8/5 ms stubs).


In [ ]:
import numpy as np  # bring in numpy for the burst arithmetic
import matplotlib  # bring in matplotlib core to pin the headless backend
matplotlib.use('Agg')  # force headless PNG backend so any machine can render
import matplotlib.pyplot as plt  # bring in pyplot for diagram + burst bars
import matplotlib.patches as patches  # bring in patches for rounded stage boxes
np.random.seed(23)  # fix the seed for reproducibility
slots, arrivals = 4, 8  # set the admission bound and the burst size
served = min(slots, arrivals)  # seat what fits: 4 get 200
shed = arrivals - served  # turn the rest away: 4 get 429, never 500
print('burst: 8 arrivals vs 4 slots -> 4 x200 + 4 x429, zero 500s')  # print the shedding contract
print('401 comes from auth, 429 from limits/queue, 200 from backends')  # print the status-code map
fig, ax = plt.subplots(2, 1, figsize=(11, 5))  # make two rows: pipeline diagram + burst bars
ax[0].set_xlim(0, 10)  # fix horizontal room for six stages
ax[0].set_ylim(0, 1.6)  # fix vertical room for boxes plus captions
ax[0].axis('off')  # hide axes because this is a diagram
ax[0].set_title('Gateway pipeline: each stage fails closed with a shaped status', fontsize=11)  # title the diagram
ax[1].bar(['200 served', '429 shed', '500 crashed'], [served, shed, 0], color=['#16a34a', '#f59e0b', '#ef4444'])  # draw the burst outcome bars
ax[1].set_ylabel('requests')  # label the y-axis
ax[1].set_title('Overload: the semaphore is a perfect guillotine (8 vs 4)')  # title the lesson
ax[1].grid(True, alpha=0.3, axis='y')  # add a light grid
plt.tight_layout()  # avoid label overlap
plt.show()  # render diagram plus simulation


## Vocabulary, one section each

Quick map — jump to whatever feels fuzzy. §4 guards the door, §5 saves work, §6 picks the chef, §7 keeps receipts.

| Term | One-liner | Section |
|---|---|---|
| Gateway | The pipeline (auth, then limits, then backends) every request passes | §4 |
| Auth (401) | Bearer-token check; strangers rejected before any work | §4 |
| Rate limit (429) | Per-key budget (lab: 5 per window); excess shed, not served | §4 |
| Admission / queue (429) | Bounded concurrent slots (lab: 4); full house sheds, never 500 | §4 |
| Cache | Exact-prompt memory: repeats return stored answer, zero backend cost | §5 |
| Batch | One shared 8 ms warm-up + 1 ms/item: k prompts cost 8+k ms | §5 |
| Router | Path-based backend picker: /generate → LLM, /batch → batched LLM, /caption → vision | §6 |
| Backend | The stub model: LLM 8 ms, vision 5 ms per call | §6 |
| Metrics | Per-route counters + mean latencies (the receipts) | §7 |
| Trace | Per-request ID + span log (investigate order by order) | §7 |
| Ledger | One row per request (phase, route, status, ms) | §7 |


### §4 Gateway, auth, limit, and queue — the three bouncers

**In one sentence: auth rejects strangers with 401, the rate limit (5 per key) sheds repeat over-askers with 429, and admission (4 concurrent slots) sheds overflow with 429 — every rejection happens BEFORE the backend burns milliseconds.**

Order matters: 401 before 429 before compute, so abuse is cheapest to refuse. Watch it: the six-stage pipeline diagram plus the shed/outcome table — 7 sends on a 5-budget key give 5 × 200 + 2 × 429, and an 8-burst on 4 slots gives 4 × 200 + 4 × 429 with zero 500s.


In [ ]:
import numpy as np  # reuse numpy for the shed table
import matplotlib.pyplot as plt  # reuse pyplot for pipeline + outcome bars
import matplotlib.patches as patches  # reuse patches for the stage boxes
np.random.seed(23)  # fix the seed for reproducibility
budget, sends = 5, 7  # set the per-key budget and the probe sends
ok_n, shed_n = min(budget, sends), max(0, sends - budget)  # split 7 sends into 5 served + 2 shed
print('rate probe: 7 sends on budget 5 ->', ok_n, 'x200 +', shed_n, 'x429')  # print the limit outcome
print('burst probe: 8 arrivals on 4 slots -> 4 x200 + 4 x429, zero 500s')  # print the admission outcome
print('unauth probe: missing token -> 401 before any backend millisecond')  # print the auth outcome
fig, ax = plt.subplots(2, 1, figsize=(11, 5))  # make two rows: stages + outcomes
ax[0].set_xlim(0, 10)  # fix horizontal room for six stages
ax[0].set_ylim(0, 1.6)  # fix vertical room
ax[0].axis('off')  # hide axes: diagram
ax[0].set_title('Gateway order: auth(401) -> limit(429) -> queue(429) -> cache -> router -> backend', fontsize=10)  # title the order rule
labels = ['auth\n401?', 'limit\n429?', 'queue\n429?', 'cache\nhit?', 'router\n3 ways', 'backend\n8/5ms']  # name the six stages
fixes = ['stops abuse', 'stops bursts', 'stops overload', 'saves repeats', 'picks chef', 'cooks answer']  # note what each rescues
xs = [0.2 + i * 1.63 for i in range(6)]  # precompute the six box left edges
print('stages:', len(labels), '(left to right, fail closed)')  # print the stage count
ax[0].text(0.925, 0.85, labels[0], ha='center', va='center', fontsize=7, weight='bold', bbox=dict(boxstyle='round', facecolor='#e0e7ff', edgecolor='#4338ca'))  # draw the auth box
ax[0].text(2.555, 0.85, labels[1], ha='center', va='center', fontsize=7, weight='bold', bbox=dict(boxstyle='round', facecolor='#e0e7ff', edgecolor='#4338ca'))  # draw the limit box
ax[0].text(4.185, 0.85, labels[2], ha='center', va='center', fontsize=7, weight='bold', bbox=dict(boxstyle='round', facecolor='#e0e7ff', edgecolor='#4338ca'))  # draw the queue box
ax[0].text(5.815, 0.85, labels[3], ha='center', va='center', fontsize=7, weight='bold', bbox=dict(boxstyle='round', facecolor='#e0e7ff', edgecolor='#4338ca'))  # draw the cache box
ax[0].text(7.445, 0.85, labels[4], ha='center', va='center', fontsize=7, weight='bold', bbox=dict(boxstyle='round', facecolor='#e0e7ff', edgecolor='#4338ca'))  # draw the router box
ax[0].text(9.075, 0.85, labels[5], ha='center', va='center', fontsize=7, weight='bold', bbox=dict(boxstyle='round', facecolor='#e0e7ff', edgecolor='#4338ca'))  # draw the backend box
ax[0].text(5, 0.32, 'each stage rescues: abuse | bursts | overload | repeats | routing | answers', ha='center', fontsize=8, style='italic')  # caption the rescue row
ax[1].bar(['200 ok', '429 shed', '401 denied'], [ok_n + 4, shed_n + 4, 1], color=['#16a34a', '#f59e0b', '#ef4444'])  # draw combined probe outcomes
ax[1].set_ylabel('requests')  # label the y-axis
ax[1].set_title('Probes combined: limits shed, auth denies, backends serve')  # title the outcome mix
ax[1].grid(True, alpha=0.3, axis='y')  # add a light grid
plt.tight_layout()  # snug the layout
plt.show()  # render both panels


### §5 Cache and batch — stop paying twice

**In one sentence: the cache returns exact repeats with zero backend cost (`cached=true`), and batching shares one 8 ms warm-up across k prompts (8+k ms instead of k×8 ms), so per-item cost falls from 8 ms to about 3 ms at k=4.**

Cache helps only repeats; limits only refuse work — neither makes one fresh prompt cheaper. Only batching lowers fresh-prompt latency, by amortizing the fixed cost. Watch it: hit-vs-miss bars (8.0 vs ~0.2 ms) plus serial-vs-batch bars (32 vs 12 ms for 4).


In [ ]:
import numpy as np  # reuse numpy for the savings math
import matplotlib.pyplot as plt  # reuse pyplot for savings bars
np.random.seed(23)  # fix the seed for reproducibility
single_ms, k = 8.0, 4  # set the stub LLM cost per prompt and the batch size
serial = k * single_ms  # price four singles back to back: 32.0ms
batched = 8.0 + 1.0 * k  # price one shared warm-up plus 1ms each: 12.0ms
print('serial 4x:', round(serial, 1), 'ms vs batched 4x:', round(batched, 1), 'ms')  # print the batching dividend
print('per-item: serial 8.0 ms vs batched', round(batched / k, 2), 'ms')  # print sharing the warm-up wins
prompts = ['a', 'b', 'a', 'c', 'b']  # set five requests with two repeats
seen = set()  # track prompts answered so far
hits = 0  # zero the repeat counter
misses = 0  # zero the fresh counter
order = [0, 1, 2, 3, 4]  # walk request positions in arrival order
first = [prompts[i] not in prompts[:i] for i in order]  # flag first-sight vs repeat per position
hits = sum(1 for f in first if not f)  # count repeats: positions 3 and 5
misses = sum(1 for f in first if f)  # count fresh: positions 1, 2, 4
print('toy traffic:', prompts, '(repeats at positions 3 and 5)')  # print the request mix
print('toy result: hits =', hits, '| misses =', misses, '(2 of 5 free)')  # print the counted outcome
fig, ax = plt.subplots(1, 2, figsize=(11, 4))  # make two panels: batch bars + cache bars
ax[0].bar(['4x serial', 'batch of 4'], [serial, batched], color=['#9ca3af', '#16a34a'])  # draw total ms per strategy
ax[0].set_ylabel('total ms')  # label the y-axis
ax[0].set_title('Batching shares one warm-up (32 -> 12ms)')  # title the mechanism
ax[0].grid(True, alpha=0.3, axis='y')  # add a light grid
ax[1].bar(['miss (backend)', 'hit (cache)'], [8.0, 0.2], color=['#f59e0b', '#16a34a'])  # draw ms per request type
ax[1].set_ylabel('ms per request')  # label the y-axis
ax[1].set_title('Cache: repeats skip the backend')  # title the lesson
ax[1].grid(True, alpha=0.3, axis='y')  # add a light grid
plt.tight_layout()  # snug the layout
plt.show()  # render both panels
print('cache hits: 2 / 5 (repeats are free; miss 8ms, hit about 0.2ms)')  # print the hit-rate moral


### §6 Router — the right chef per order

**In one sentence: the router picks the backend by path — `/generate` → LLM stub (8 ms), `/batch` → batched LLM stub (8+k ms), `/caption` → vision stub (5 ms tag lookup) — so text, bulk text, and image orders never block each other's ovens.**

Routing is what makes batching and vision coexist: bulk orders share one warm-up while vision skips the LLM entirely. Unknown tags fall back to `'unknown scene'` (closed vocabulary, honest miss, still 200). Watch it: the route-selection table plus one call traced per route.


In [ ]:
import numpy as np  # reuse numpy for the route timing table
import matplotlib.pyplot as plt  # reuse pyplot for the route latency bars
np.random.seed(23)  # fix the seed for reproducibility
routes = ['/generate', '/batch x3', '/caption']  # name the three serving routes
cost = [8.0, 11.0, 5.0]  # set stub cost per route: 8ms, 8+3ms, 5ms
print('route costs (ms):', dict(zip(routes, cost)))  # print the per-route price list
print('/generate reverses words + [ntok]; /caption maps pitch/chart/ocr tags')  # print the stub behaviours
print('unknown tag -> unknown scene (closed vocabulary, still 200)')  # print the fallback contract
fig, ax = plt.subplots(figsize=(7, 4))  # make a figure for the route bars
ax.bar(routes, cost, color=['#3b82f6', '#16a34a', '#8b5cf6'])  # draw ms per route
ax.set_ylabel('ms per call')  # label the y-axis
ax.set_title('Router: each order pays its own chef (batch cheapest/item)')  # title the routing win
ax.grid(True, alpha=0.3, axis='y')  # add a light grid
plt.tight_layout()  # snug the layout
plt.show()  # render the bars


### §7 Metrics, trace, and ledger — the receipts

**In one sentence: metrics count outcomes per (route, status) plus mean latency, the trace logs one span dict per request (id, route, status, ms), and the ledger writes one CSV row per request — including rejections, because a spike of 401s (broken client/attack) or 429s (under-provisioning) is signal.**

Unmetered rejections make overloads and abuse invisible. Watch it: one sample span dict, the counter table, and the ledger-outcome bars.


In [ ]:
import numpy as np  # reuse numpy for the outcome mix
import matplotlib.pyplot as plt  # reuse pyplot for the outcome bars
np.random.seed(23)  # fix the seed for reproducibility
span = {'id': 7, 'route': 'llm', 'status': 200, 'ms': 8.421}  # build one sample trace span
print('sample span:', span)  # print the per-request trace shape
counts = {('llm', 200): 9, ('llm', 429): 1, ('llm', 401): 1, ('batch', 200): 1, ('vlm', 200): 1}  # set the mix-phase counters
print('counters:', counts)  # print the metrics table
print('mean llm latency about 8ms; rejects metered at about 0.1ms each')  # print the latency moral
labels = ['200', '429', '401']  # name the status classes
vals = [11, 1, 1]  # count ledger rows per class in the toy mix
print('ledger mix: 200 x', vals[0], '+ 429 x', vals[1], '+ 401 x', vals[2])  # print the outcome mix
fig, ax = plt.subplots(figsize=(7, 4))  # make a figure for the outcome bars
ax.bar(labels, vals, color=['#16a34a', '#f59e0b', '#ef4444'])  # draw requests per status
ax.set_ylabel('requests')  # label the y-axis
ax.set_title('Ledger outcomes: serve, shed, deny (all metered)')  # title the receipts rule
ax.grid(True, alpha=0.3, axis='y')  # add a light grid
plt.tight_layout()  # snug the layout
plt.show()  # render the bars


## How it all fits together — the system map

One request walks every § station in gateway order. **Auth → limit → queue** (§4): 401/429/429 before compute. **Cache** (§5): exact repeats free. **Router → backend** (§6): LLM / batch / vision at 8, 8+k, 5 ms. **Metrics + trace + ledger** (§7): every outcome counted, including rejections. The map pins each term; the numbers walk one cached request and one shed request side by side.


In [ ]:
import matplotlib.pyplot as plt  # only pyplot: this cell is a pure diagram
fig, ax = plt.subplots(figsize=(11, 3.5))  # create a wide figure for the gateway pipeline
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 16)  # fix horizontal placement range
ax.set_ylim(0, 4)  # fix vertical placement range
ax.text(8, 3.6, 'REQUEST: auth -> limit -> queue -> cache -> router -> backend -> receipts', ha='center', fontsize=9, fontweight='bold')  # order banner line
ax.text(2, 2.4, 'auth/limit\nqueue §4', ha='center', va='center', fontsize=8, bbox=dict(boxstyle='round', facecolor='lightblue'))  # draw the guards box
ax.text(5.4, 2.4, 'cache §5', ha='center', va='center', fontsize=8, bbox=dict(boxstyle='round', facecolor='lightblue'))  # draw the cache box
ax.text(8.2, 2.4, 'router §6', ha='center', va='center', fontsize=8, bbox=dict(boxstyle='round', facecolor='lightblue'))  # draw the router box
ax.text(11, 2.4, 'backend §6', ha='center', va='center', fontsize=8, bbox=dict(boxstyle='round', facecolor='lightblue'))  # draw the backend box
ax.text(14, 2.4, 'metrics §7', ha='center', va='center', fontsize=8, bbox=dict(boxstyle='round', facecolor='lightblue'))  # draw the receipts box
ax.text(8, 1.2, 'CACHED: 200 in ~0.2ms (no backend) | SHED: 429 in ~0.1ms (no backend)', ha='center', fontsize=9, fontweight='bold', color='darkred')  # red = cheap-exit banner
ax.text(8, 0.5, 'rejections metered too: 401/429 spikes are debugging signal', ha='center', fontsize=9, style='italic')  # receipts reminder line
ax.set_title('System map: request->auth->limit->queue->cache->router->backend->metrics')  # diagram title
plt.tight_layout()  # snug the layout
plt.show()  # render the map


In [ ]:
import numpy as np  # reuse numpy for the two-request walkthrough
np.random.seed(23)  # fix the seed for reproducibility
cached_ms, shed_ms, fresh_ms = 0.2, 0.1, 8.0  # set ms per outcome: hit, shed, fresh LLM call
print('cached request: auth ok -> limit ok -> queue ok -> HIT -> 200 in', cached_ms, 'ms')  # walk the repeat
print('shed request: auth ok -> limit FULL -> 429 in', shed_ms, 'ms (backend never runs)')  # walk the over-budget send
print('fresh request: auth ok -> limit ok -> queue ok -> miss -> LLM 200 in', fresh_ms, 'ms')  # walk the normal call
print('each walk appends one ledger row + one trace span + one counter bump')  # print the receipts rule
print('lab echo: mix has exactly one 401 and one 429; burst sheds with zero 500s')  # print the lab twins


## Bridge to the lab

Open `notebook.ipynb` now. Its cells implement exactly the map above (§8) with a FastAPI app driven by TestClient (in-process, no ports):

1. **Goal + diagram** — map each which-layer-rescues-which-failure pair to the restaurant stations (§4–§7).
2. **Setup + knobs** — memorize the policy numbers: 1 token, budget 5, 4 slots (§4).
3. **Backends + app** — `llm_backend` (8 ms), `vlm_backend` (5 ms), gate order auth→limit→admission (§4), each route (§6), the `note` observability write (§7); batch math is the shared-oven demo (§5).
4. **Contracts** — each assert is one promise: 401s, cache hit + consistency (§5), batch < 4× single (§5), 5-in/2-shed (§4), router tag (§6), /gpu + /metrics shapes (§7).
5. **Load ledger** — scripted mix (fresh + repeat + batch + vision + unauthenticated + budget probe) plus the barrier-aligned 8-thread burst (4 ×200 + 4 ×429, zero 500s) (§4, §7).
6. **Summarize + figure** — per-phase table to `platform.csv` + `summary.csv` + `platform.png`; mix has exactly one 401 and one 429 (§7).
7. **Cleanup/next** — swap-out map (vLLM for stubs, Redis for CACHE, OTel for TRACE, DCGM for /gpu) and the P24 serving shape (/report + /tracks).


## Self-check — predict first, then check

> Retrieval rule: cover every *Answer* below, write your prediction (a number, a shape, or one sentence), run the § cell it cites, then uncover. The last question in each set has no printed answer — do it from the map.



1. **A burst of 8 hits 4 slots and yields 4 ×200 + 4 ×429 with zero 500s. Why is shedding with 429 better than queueing forever (§4)?**

*Answer: bounded admission keeps latency predictable and the server alive; 429 tells the client to retry later, while unbounded queueing would time out everywhere and risk crashing (500) under load.*

2. **Cache helps repeats and limits protect, but only batching lowers per-item latency. Why (§5)?**

*Answer: cache skips work only for exact repeats and limits only refuse work — neither makes one fresh prompt cheaper. Batching amortizes the fixed 8 ms warm-up over k items, so per-item cost falls from 8 ms to about 3 ms at k=4 — the §8 trace prices all three side by side.*

3. **Why meter rejected requests (401/429) in metrics and traces too (§7)?**

*Answer: rejections are signal: a spike of 401s means a broken client or attack, a spike of 429s means under-provisioning. Unmetered rejections make overloads and abuse invisible.*
